In [ ]:
# Run once
!pip install datasets pandas networkx tqdm matplotlib seaborn python-louvain scipy

In [ ]:
# Imports and parameters
from datasets import load_dataset
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import defaultdict, deque
import os
from community import community_louvain
from scipy.stats import mannwhitneyu

sns.set(style="whitegrid", context="talk")

# --- EDIT THESE PARAMETERS ---
SUBREDDITS = ["politics", "worldnews", "technology"]   # pick 1-4 subreddits
START_YEAR = 2023
START_MONTH = 1
END_YEAR = 2023
END_MONTH = 1
# We'll filter by subreddit only; time filter below uses created_utc if desired.
MAX_COMMENTS = 100000   # total comments to stream (tune: 50k-200k)
MAX_POSTS = None         # optional: limit number of distinct posts (None = unlimited)
RANDOM_SEED = 42
OUT_DIR = "hf_stream_slice"
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)
print("Parameters:")
print("Subreddits:", SUBREDDITS)
print("Max comments (approx):", MAX_COMMENTS)
print("Output dir:", OUT_DIR)

In [ ]:
# Stream the HuggingFace Pushshift dataset and collect comments for chosen subreddits.
# NOTE: this iterates over the server-side dataset and stops after MAX_COMMENTS are collected.

dataset_id = "fddemarco/pushshift-reddit-comments"  # Hugging Face dataset id (Pushshift comments)
print("Loading dataset (streaming)...", dataset_id)

stream = load_dataset(dataset_id, split="train", streaming=True)  # streaming iterator

rows = []
post_set = set()
count = 0
max_comments = MAX_COMMENTS
max_posts = MAX_POSTS

for item in tqdm(stream, total=None, desc="Streaming comments"):
    # item is a dict with fields like: id, parent_id, link_id, author, subreddit, created_utc, score, controversiality, body
    try:
        sub = item.get("subreddit")
        if sub is None:
            continue
        if sub.lower() not in [s.lower() for s in SUBREDDITS]:
            continue
        # optionally filter by time if item has created_utc and you want a specific month
        # e.g., only include comments in January 2023
        # if 'created_utc' in item:
        #     utc = int(item['created_utc'])
        #     # convert & filter if desired...

        # Collect row
        rows.append({
            "id": item.get("id"),
            "parent_id": item.get("parent_id"),
            "link_id": item.get("link_id"),
            "author": item.get("author"),
            "subreddit": item.get("subreddit"),
            "created_utc": item.get("created_utc"),
            "score": item.get("score"),
            "controversiality": int(item.get("controversiality") or 0),
            "body": item.get("body")
        })
        count += 1
        # track posts
        if item.get("link_id"):
            # normalize link id without prefix
            lk = str(item.get("link_id")).replace("t3_", "")
            post_set.add(lk)
        # stopping conditions
        if max_comments and count >= max_comments:
            break
        if max_posts and len(post_set) >= max_posts:
            break
    except Exception as e:
        # skip problematic row
        continue

print(f"Collected {len(rows)} comments across {len(post_set)} posts (approx).")
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "comments_stream_sample.csv"), index=False)
df.head()

In [ ]:
# Preprocess IDs: strip t1_/t3_ prefixes and ensure strings
def strip_prefix(x):
    if pd.isna(x): return x
    s = str(x)
    return s.replace("t1_", "").replace("t3_", "")

df['id'] = df['id'].astype(str)
df['parent_id_raw'] = df['parent_id'].astype(str)
df['parent_id_clean'] = df['parent_id_raw'].apply(lambda x: strip_prefix(x) if x not in ["None", "nan"] else "")
df['link_id_clean'] = df['link_id'].astype(str).apply(lambda x: strip_prefix(x) if x not in ["None", "nan"] else "")
df['author'] = df['author'].astype(str)
df['created_utc'] = pd.to_numeric(df['created_utc'], errors='coerce')
df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0).astype(int)
df['controversiality'] = df['controversiality'].fillna(0).astype(int)

print("After cleaning:", df.shape)
df[['id','parent_id_clean','link_id_clean','author','subreddit','score','controversiality']].head()

In [ ]:
# Compute per-post aggregates (num_comments, num_controversial_comments, mean_score)
post_grp = df.groupby('link_id_clean').agg(
    num_comments = ('id','count'),
    num_controversial_comments = ('controversiality','sum'),
    mean_score = ('score','mean')
).reset_index().rename(columns={'link_id_clean':'post_id'})

# Derive a post-level controversy score and label:
# We do this because many comment-only datasets do not include the submission's explicit controversiality flag.
# Heuristic (transparent): controversy_score = fraction_controversial_comments + 0.3*(num_comments/(mean_score+1))
# Then label top 20% posts by controversy_score as controversial (1) and bottom 80% as non-controversial (0)
post_grp['frac_contro'] = post_grp['num_controversial_comments'] / post_grp['num_comments']
post_grp['controversy_score'] = post_grp['frac_contro'] + 0.3*(post_grp['num_comments'] / (post_grp['mean_score'].abs()+1))
# handle NaNs
post_grp['controversy_score'] = post_grp['controversy_score'].fillna(0)
# label threshold: top 20% controversial
pct = 0.20
threshold = post_grp['controversy_score'].quantile(1-pct)
post_grp['post_controversial'] = (post_grp['controversy_score'] >= threshold).astype(int)

print("Posts aggregated:", post_grp.shape)
post_grp.head()

In [ ]:
# Map post label to comments
post_label_map = post_grp.set_index('post_id')['post_controversial'].to_dict()
df['post_controversial'] = df['link_id_clean'].map(post_label_map).fillna(0).astype(int)

# Save interim data
df.to_csv(os.path.join(OUT_DIR, "comments_with_post_labels.csv"), index=False)
post_grp.to_csv(os.path.join(OUT_DIR, "post_aggregates.csv"), index=False)

print("Comments with post label saved. Example distribution:")
print(df['post_controversial'].value_counts())

In [ ]:
from collections import defaultdict, deque

def compute_cascade_metrics_for_post(post_id, comments_df):
    cs = comments_df[comments_df['link_id_clean']==str(post_id)].copy()
    if cs.empty:
        return None
    # Prepare children map
    ids = cs['id'].astype(str).tolist()
    id_set = set(ids)
    children = defaultdict(list)
    for _, r in cs.iterrows():
        cid = str(r['id'])
        pid = r['parent_id_clean'] if pd.notna(r['parent_id_clean']) else ''
        # if parent id is empty or equals the post id, that means a top-level comment replying to submission
        children[pid].append(cid)
    # identify roots: comments whose parent is the post (parent == post_id) or parent not in id_set
    roots = [cid for cid in ids if (cs.loc[cs['id']==cid,'parent_id_clean'].values[0] == str(post_id))
             or (cs.loc[cs['id']==cid,'parent_id_clean'].values[0] not in id_set)]
    # BFS for depth
    max_depth = 0
    q = deque()
    for r in roots:
        q.append((r,1))
    visited = set()
    while q:
        node, depth = q.popleft()
        max_depth = max(max_depth, depth)
        visited.add(node)
        for ch in children.get(node, []):
            if ch not in visited:
                q.append((ch, depth+1))
    total_nodes = len(id_set)
    unique_users = cs['author'].nunique() if 'author' in cs.columns else np.nan
    # lifespan (seconds)
    if 'created_utc' in cs.columns:
        times = cs['created_utc'].dropna().astype(float).sort_values()
        lifespan = float(times.max() - times.min()) if len(times) > 1 else 0.0
    else:
        lifespan = np.nan
    # branching: edges/internals (approx)
    internal_nodes = sum(1 for k,v in children.items() if k in id_set and len(v)>0)
    branching = (total_nodes - 1) / internal_nodes if internal_nodes > 0 else np.nan
    return {
        'post_id': post_id,
        'cascade_size': total_nodes,
        'max_depth': max_depth,
        'unique_users': unique_users,
        'lifespan_sec': lifespan,
        'branching': branching
    }

In [ ]:
# Compute metrics for all posts in post_grp (or optionally subset top-K by num_comments)
posts = post_grp['post_id'].astype(str).tolist()
metrics = []
for pid in tqdm(posts, desc="Computing post cascades"):
    m = compute_cascade_metrics_for_post(pid, df)
    if m:
        m['post_controversial'] = int(post_label_map.get(pid, 0))
        metrics.append(m)
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(os.path.join(OUT_DIR, "cascade_metrics.csv"), index=False)
metrics_df.head()

In [ ]:
# Boxplots (use log scale for heavy-tailed distributions)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.boxplot(data=metrics_df, x='post_controversial', y='cascade_size')
plt.yscale('log')
plt.xlabel('post_controversial (1=yes, 0=no)')
plt.title('Cascade size: controversial vs non')

plt.subplot(1,2,2)
sns.boxplot(data=metrics_df, x='post_controversial', y='max_depth')
plt.yscale('log')
plt.xlabel('post_controversial')
plt.title('Cascade max depth: controversial vs non')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "cascade_size_depth_boxplots.png"), dpi=150, bbox_inches='tight')
plt.show()

# Mann-Whitney U tests (non-parametric)
c1 = metrics_df[metrics_df['post_controversial']==1]['cascade_size'].dropna()
c0 = metrics_df[metrics_df['post_controversial']==0]['cascade_size'].dropna()
if len(c1)>0 and len(c0)>0:
    stat, p = mannwhitneyu(c1, c0, alternative='two-sided')
    print("Mann-Whitney U: cascade_size stat=%.3f p=%.3e" % (stat, p))

In [ ]:
# Clean parent ids so they match comment ids
df["parent_clean"] = df["parent_id"].str.replace("t1_", "", regex=False)
df["parent_clean"] = df["parent_clean"].str.replace("t3_", "", regex=False)

df.head()

In [ ]:
import networkx as nx
from tqdm import tqdm

G = nx.DiGraph()

id_set = set(df["id"])

for _, row in tqdm(df.iterrows(), total=len(df)):

    parent = row["parent_clean"]
    child = row["id"]

    if parent in id_set:
        G.add_edge(parent, child)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
import networkx as nx
from tqdm.auto import tqdm

# Build directed graph from the dense targeted dataset
G = nx.DiGraph()

# Map for parent author lookup within the dense set
comment_author_map = pd.Series(df_dense['author'].values, index=df_dense['id'].astype(str)).to_dict()

edge_count = 0
for _, row in tqdm(df_dense.iterrows(), total=df_dense.shape[0], desc='Building dense user edges'):
    child_author = row['author']
    parent_raw = str(row['parent_id'])

    # Clean the parent_id prefix
    if '_' in parent_raw:
        parts = parent_raw.split('_', 1)
        prefix, parent_id = parts[0], parts[1]
        # We only link comment-to-comment (t1) replies for the user network
        if prefix == 't1':
            parent_author = comment_author_map.get(parent_id)
            if parent_author and child_author and child_author != '[deleted]' and parent_author != '[deleted]':
                G.add_edge(child_author, parent_author)
                edge_count += 1

print(f'User graph nodes: {G.number_of_nodes()}')
print(f'User graph edges: {G.number_of_edges()}')

# Update undirected version for metrics
G_und = G.to_undirected() if G.number_of_nodes() > 0 else nx.Graph()

In [ ]:
# Basic network metrics
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
density = nx.density(G) if num_nodes>0 else 0
G_und = G.to_undirected() if num_nodes>0 else nx.Graph()
avg_clustering = nx.average_clustering(G_und) if num_nodes>0 else np.nan

print("Nodes:", num_nodes, "Edges:", num_edges, "Density:", density, "Avg clustering:", avg_clustering)

# Degree distribution histogram (log-log)
deg_seq = [d for _,d in G_und.degree()] if num_nodes>0 else []
plt.figure(figsize=(6,4))
plt.hist(deg_seq, bins=50)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Degree'); plt.ylabel('Frequency')
plt.title('Degree distribution (user graph)')
plt.savefig(os.path.join(OUT_DIR,'degree_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# gemini
from collections import Counter
from tqdm.auto import tqdm
from datasets import load_dataset

# Pass 1: Scan 100k comments to find the most active 'Seed' posts
dataset_id = "fddemarco/pushshift-reddit-comments"
stream = load_dataset(dataset_id, split="train", streaming=True)

post_counter = Counter()
MAX_SCAN = 100000
count = 0

print(f"Scanning first {MAX_SCAN} comments to find active threads in {SUBREDDITS}...")

for item in tqdm(stream, total=MAX_SCAN, desc="Identifying active posts"):
    sub = item.get("subreddit")
    if sub and sub.lower() in [s.lower() for s in SUBREDDITS]:
        lk = item.get("link_id")
        if lk:
            post_counter[lk] += 1
        count += 1

    if count >= MAX_SCAN:
        break

# Get the top 100 most active post IDs
TOP_POSTS = [p for p, c in post_counter.most_common(100)]
print(f"\nIdentified {len(TOP_POSTS)} seed posts.")
print("Top 5 most active IDs:", post_counter.most_common(5))

In [ ]:
# Centrality on the dense graph
if G.number_of_nodes() > 0:
    deg_cent = nx.degree_centrality(G)

    # Betweenness centrality: approximate if big
    if G.number_of_nodes() > 2000:
        bet_cent = nx.betweenness_centrality(G, k=200, seed=RANDOM_SEED)
    else:
        bet_cent = nx.betweenness_centrality(G)

    close_cent = nx.closeness_centrality(G)
    try:
        eig_cent = nx.eigenvector_centrality_numpy(G_und)
    except Exception:
        eig_cent = {}

    # Assemble centrality table
    users = list(deg_cent.keys())
    central_df = pd.DataFrame({
        'user': users,
        'degree_centrality': [deg_cent.get(u,0) for u in users],
        'betweenness': [bet_cent.get(u,0) for u in users],
        'closeness': [close_cent.get(u,0) for u in users],
        'eigenvector': [eig_cent.get(u,0) for u in users]
    })
    central_df = central_df.sort_values('degree_centrality', ascending=False).reset_index(drop=True)
    display(central_df.head(20))
else:
    print("Graph is empty. Please check Pass 1 and Pass 2 results.")

In [ ]:
import pandas as pd
import networkx as nx
from tqdm import tqdm

# 1. Check how many parents actually exist in our local 'id' set
id_set = set(df['id'].astype(str).unique())

# Standardize parent_id to remove prefixes for comparison
def get_clean_id(val):
    if pd.isna(val) or str(val).lower() == 'none': return None
    s = str(val)
    if '_' in s: return s.split('_')[1]
    return s

df['parent_clean_diag'] = df['parent_id'].apply(get_clean_id)
df['id_clean_diag'] = df['id'].apply(get_clean_id)

clean_id_set = set(df['id_clean_diag'].unique())
matches = df[df['parent_clean_diag'].isin(clean_id_set)]

print(f"Total comments: {len(df)}")
print(f"Comments with a parent present in this sample: {len(matches)}")

# 2. Re-build the graph using these cleaned IDs
G = nx.DiGraph()
for _, row in matches.iterrows():
    # Link child author to parent author
    child_author = row['author']
    parent_id = row['parent_clean_diag']

    # Find the author of the parent comment
    parent_row = df[df['id_clean_diag'] == parent_id]
    if not parent_row.empty:
        parent_author = parent_row.iloc[0]['author']
        if child_author != '[deleted]' and parent_author != '[deleted]':
            G.add_edge(child_author, parent_author)

print(f"New Graph - Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

if G.number_of_edges() == 0:
    print('NOTE: Still 0 edges. This means in this specific 20k slice, no one replied to anyone else also in the slice.')
else:
    display(central_df.head())

In [ ]:
# Run Louvain on undirected graph
if G_und.number_of_nodes() > 0 and G_und.number_of_edges() > 0:
    partition = community_louvain.best_partition(G_und)
    num_communities = len(set(partition.values()))
    mod = community_louvain.modularity(partition, G_und)
    print("Detected communities:", num_communities, "Modularity:", mod)
    # Save partition mapping
    part_df = pd.DataFrame(list(partition.items()), columns=['user','community'])
    part_df.to_csv(os.path.join(OUT_DIR,'user_communities.csv'), index=False)
else:
    partition = {}
    mod = np.nan
    part_df = pd.DataFrame()

In [ ]:
# Map users to communities (if present)
user_comm = partition

comm_spread = []
for _, row in tqdm(metrics_df.iterrows(), total=metrics_df.shape[0], desc="Community spread per cascade"):
    pid = str(row['post_id'])
    cs = df[df['link_id_clean']==pid]
    if cs.empty:
        continue
    authors = cs['author'].dropna().astype(str).unique().tolist()
    comms = set([user_comm.get(a, -1) for a in authors])
    num_known = len([c for c in comms if c!=-1])
    frac_known = num_known / (len(authors) if len(authors)>0 else 1)
    comm_spread.append({
        'post_id': pid,
        'cascade_size': row['cascade_size'],
        'num_communities_known': num_known,
        'frac_communities_known': frac_known,
        'post_controversial': int(row['post_controversial'])
    })

comm_spread_df = pd.DataFrame(comm_spread)
comm_spread_df.to_csv(os.path.join(OUT_DIR,'community_spread.csv'), index=False)
comm_spread_df.head()

In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(data=comm_spread_df, x='post_controversial', y='frac_communities_known')
plt.xlabel('post_controversial (1/0)')
plt.ylabel('fraction of unique communities (known)')
plt.title('Community spread by controversy label')
plt.savefig(os.path.join(OUT_DIR,'community_spread_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

v1 = comm_spread_df[comm_spread_df['post_controversial']==1]['frac_communities_known'].dropna()
v0 = comm_spread_df[comm_spread_df['post_controversial']==0]['frac_communities_known'].dropna()
if len(v1)>10 and len(v0)>10:
    st, p = mannwhitneyu(v1, v0, alternative='two-sided')
    print("Mann-Whitney community spread: stat=%.3f p=%.3e" % (st, p))
else:
    print("Not enough samples for robust test.")

In [ ]:
metrics_df.to_csv(os.path.join(OUT_DIR,'cascade_metrics_full.csv'), index=False)
central_df.to_csv(os.path.join(OUT_DIR,'user_centralities_full.csv'), index=False)
summary = {
    'num_comments': int(df.shape[0]),
    'num_posts': int(post_grp.shape[0]),
    'num_posts_analyzed': int(metrics_df.shape[0]),
    'num_users': int(G.number_of_nodes()),
    'num_user_edges': int(G.number_of_edges()),
    'modularity': float(mod if not np.isnan(mod) else -1)
}
pd.Series(summary).to_csv(os.path.join(OUT_DIR,'analysis_summary.csv'))
print("All outputs saved to", OUT_DIR)
summary

# Task
Increase the data volume by setting `MAX_COMMENTS` to 100,000 to better capture parent-child comment relationships in the "fddemarco/pushshift-reddit-comments" dataset. Modify the data collection process to focus on a specific set of active post IDs rather than a random global stream to ensure higher thread density. Finally, re-execute the network analysis, including graph construction and centrality calculations, on this denser dataset to improve the reliability of the social network metrics.

## Increase Data Volume

### Subtask:
Increase the comment collection limit to 100,000 to improve the capture of parent-child relationships.


**Reasoning**:
The current data slice of 20,000 comments resulted in 0 edges in the user graph, meaning no parent-child relationships were captured. Increasing the data volume to 100,000 comments as instructed will significantly improve the likelihood of capturing conversation threads and building a meaningful network.



In [ ]:
# Imports and parameters
from datasets import load_dataset
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from collections import defaultdict, deque
import os
from community import community_louvain
from scipy.stats import mannwhitneyu

sns.set(style="whitegrid", context="talk")

# --- EDIT THESE PARAMETERS ---
SUBREDDITS = ["politics", "worldnews", "technology"]   # pick 1-4 subreddits
START_YEAR = 2023
START_MONTH = 1
END_YEAR = 2023
END_MONTH = 1
# Increased limit to improve parent-child relationship capture
MAX_COMMENTS = 100000   # Updated from 20000 to 100000
MAX_POSTS = None         # optional: limit number of distinct posts (None = unlimited)
RANDOM_SEED = 42
OUT_DIR = "hf_stream_slice"
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(RANDOM_SEED)
print("Parameters Updated:")
print("Subreddits:", SUBREDDITS)
print("Max comments (approx):", MAX_COMMENTS)
print("Output dir:", OUT_DIR)

**Reasoning**:
With the parameter MAX_COMMENTS successfully updated to 100,000, I will now execute the data collection step to stream the larger volume of comments from the Hugging Face dataset. This will increase the density of the sampled data and improve the chances of capturing parent-child comment relationships for the network analysis.



In [ ]:
dataset_id = "fddemarco/pushshift-reddit-comments"
print("Loading dataset (streaming)...", dataset_id)

stream = load_dataset(dataset_id, split="train", streaming=True)

rows = []
post_set = set()
count = 0
max_comments = MAX_COMMENTS
max_posts = MAX_POSTS

for item in tqdm(stream, total=None, desc="Streaming comments (100k target)"):
    try:
        sub = item.get("subreddit")
        if sub is None:
            continue
        if sub.lower() not in [s.lower() for s in SUBREDDITS]:
            continue

        rows.append({
            "id": item.get("id"),
            "parent_id": item.get("parent_id"),
            "link_id": item.get("link_id"),
            "author": item.get("author"),
            "subreddit": item.get("subreddit"),
            "created_utc": item.get("created_utc"),
            "score": item.get("score"),
            "controversiality": int(item.get("controversiality") or 0),
            "body": item.get("body")
        })
        count += 1
        if item.get("link_id"):
            lk = str(item.get("link_id")).replace("t3_", "")
            post_set.add(lk)
        if max_comments and count >= max_comments:
            break
        if max_posts and len(post_set) >= max_posts:
            break
    except Exception as e:
        continue

print(f"Collected {len(rows)} comments across {len(post_set)} posts.")
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "comments_stream_100k.csv"), index=False)
df.head()"

**Reasoning**:
The previous cell failed with a SyntaxError because of a trailing double quote at the end of the `df.head()` call. I will remove this extra character to fix the script and allow the 100k comment collection to proceed.



In [ ]:
dataset_id = "fddemarco/pushshift-reddit-comments"
print("Loading dataset (streaming)...", dataset_id)

stream = load_dataset(dataset_id, split="train", streaming=True)

rows = []
post_set = set()
count = 0
max_comments = MAX_COMMENTS
max_posts = MAX_POSTS

for item in tqdm(stream, total=None, desc="Streaming comments (100k target)"):
    try:
        sub = item.get("subreddit")
        if sub is None:
            continue
        if sub.lower() not in [s.lower() for s in SUBREDDITS]:
            continue

        rows.append({
            "id": item.get("id"),
            "parent_id": item.get("parent_id"),
            "link_id": item.get("link_id"),
            "author": item.get("author"),
            "subreddit": item.get("subreddit"),
            "created_utc": item.get("created_utc"),
            "score": item.get("score"),
            "controversiality": int(item.get("controversiality") or 0),
            "body": item.get("body")
        })
        count += 1
        if item.get("link_id"):
            lk = str(item.get("link_id")).replace("t3_", "")
            post_set.add(lk)
        if max_comments and count >= max_comments:
            break
        if max_posts and len(post_set) >= max_posts:
            break
    except Exception as e:
        continue

print(f"Collected {len(rows)} comments across {len(post_set)} posts.")
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "comments_stream_100k.csv"), index=False)
display(df.head())